# Notebook 04 — CrowS-Pairs: SPS Metric + Bolukbasi Geometric Analysis

This notebook covers two analyses on the **CrowS-Pairs** dataset (Nangia et al., 2020), both across baseline, post-LoRA, and post-QLoRA conditions.

## Task 2: SPS (Stereotype Preference Score)
For each pair of sentences — one more stereotyping (`sent_more`), one less (`sent_less`) — score which one the model assigns higher log-likelihood. SPS = % of pairs where the model prefers the stereotyped sentence.

- **Ideal (unbiased model):** SPS = 50% (no preference)
- **Biased model:** SPS > 50%

## Task 3: Bolukbasi Geometric Metric on CrowS-Pairs
For each sentence pair, identify the words that **differ** between `sent_more` and `sent_less` (the "swapped" tokens). Extract contextual representations of those words from the model and measure how strongly they project onto the gender direction **g** from Notebook 03.

Key question: Do the words in the more-stereotyping sentence sit geometrically closer to the gender axis than the words in the less-stereotyping sentence?

**Geometric SPS** = % of pairs where the stereotyped changed word projects more onto g than the anti-stereotyped changed word.

---

## References
- Nangia et al. (2020). *CrowS-Pairs: A Challenge Dataset for Measuring Social Biases in Masked Language Models.* EMNLP 2020.
- Bolukbasi et al. (2016). *Man is to Computer Programmer as Woman is to Homemaker?* NeurIPS 2016.


## 1. Imports & Setup

In [ ]:
import torch
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"
LORA_PATH  = "../results/lora_adapter"
QLORA_PATH = "../results/qlora_adapter"

torch.manual_seed(42)
np.random.seed(42)
print(f"Device: {DEVICE}")

## 2. Load CrowS-Pairs (Gender Subset)

In [ ]:
print("Loading CrowS-Pairs...")
crows = load_dataset("nyu-mll/crows_pairs", split="test")

print(f"Total examples: {len(crows)}")
print(f"Bias types: {set(crows['bias_type'])}")

# Filter to gender bias only
crows_gender = crows.filter(lambda x: x["bias_type"] == "gender")
print(f"Gender examples: {len(crows_gender)}")

# Preview
ex = crows_gender[0]
print(f"\nsent_more (more stereotyping): {ex['sent_more']}")
print(f"sent_less (less stereotyping): {ex['sent_less']}")
print(f"stereo_antistereo: {ex['stereo_antistereo']}")

## 3. SPS — Stereotype Preference Score

**Definition (adapted for causal LM):**
$$\text{SPS} = \frac{1}{|P|} \sum_{(s_+, s_-) \in P} \mathbb{1}[\log P_\theta(s_+) > \log P_\theta(s_-)] \times 100$$

Where $s_+$ = `sent_more` (more stereotyping), $s_-$ = `sent_less` (less stereotyping).

Log-likelihood is normalised by sequence length to avoid penalising longer sentences.

> **Note:** CrowS-Pairs was originally designed for masked LMs using pseudo-log-likelihood. For causal LMs like OPT, we use standard causal log-likelihood (sum of log P(token | all previous tokens)), normalised by token count.

In [ ]:
def sentence_log_likelihood(model, tokenizer, sentence):
    """
    Returns the average per-token log-likelihood of the sentence under the model.
    Higher = model considers this sentence more natural/probable.
    """
    inputs = tokenizer(sentence, return_tensors="pt").to(model.device)
    n_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    # outputs.loss = mean NLL per token (cross-entropy)
    # log-likelihood = -loss
    return -outputs.loss.item()   # higher = more likely


def compute_sps(model, tokenizer, dataset, desc=""):
    """
    Compute Stereotype Preference Score.
    Returns: sps (float 0-100), per_pair results list.
    """
    prefer_stereo = 0
    records       = []

    for ex in tqdm(dataset, desc=f"SPS {desc}"):
        ll_more = sentence_log_likelihood(model, tokenizer, ex["sent_more"])
        ll_less = sentence_log_likelihood(model, tokenizer, ex["sent_less"])

        prefers_more = ll_more > ll_less
        prefer_stereo += int(prefers_more)

        records.append({
            "sent_more":      ex["sent_more"],
            "sent_less":      ex["sent_less"],
            "ll_more":        round(ll_more, 4),
            "ll_less":        round(ll_less, 4),
            "prefers_stereo": int(prefers_more),
        })

    sps = 100 * prefer_stereo / len(dataset)
    print(f"SPS {desc}: {sps:.2f}%  ({prefer_stereo}/{len(dataset)} pairs prefer stereotyped sentence)")
    print(f"  Ideal (unbiased) = 50.00%  |  Bias above baseline = {sps - 50:.2f}pp")
    return sps, records

## 4. Bolukbasi Geometric Functions (reused from Notebook 03)

In [ ]:
# ---- Gender pairs for computing g ----
GENDER_PAIRS = [
    ("he", "she"), ("him", "her"), ("his", "hers"),
    ("man", "woman"), ("men", "women"), ("boy", "girl"),
    ("male", "female"), ("father", "mother"), ("son", "daughter"),
    ("brother", "sister"), ("husband", "wife"), ("king", "queen"),
    ("prince", "princess"), ("uncle", "aunt"), ("nephew", "niece"),
]

def get_contextual_rep(model, tokenizer, word, template="The {word} works in the city."):
    text = template.format(word=word)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    word_tokens = tokenizer(" " + word, add_special_tokens=False)["input_ids"]
    input_ids   = inputs["input_ids"][0].tolist()

    positions = []
    for i in range(len(input_ids) - len(word_tokens) + 1):
        if input_ids[i:i+len(word_tokens)] == word_tokens:
            positions = list(range(i, i + len(word_tokens)))
            break

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    last_hidden = outputs.hidden_states[-1][0]
    if positions:
        return last_hidden[positions].mean(dim=0).float().cpu().numpy()
    return last_hidden.mean(dim=0).float().cpu().numpy()


def compute_gender_direction(model, tokenizer, gender_pairs):
    diffs = []
    for male_w, female_w in gender_pairs:
        vm = get_contextual_rep(model, tokenizer, male_w)
        vf = get_contextual_rep(model, tokenizer, female_w)
        diffs.append(vm - vf)
    pca = PCA(n_components=1)
    pca.fit(np.array(diffs))
    g = pca.components_[0]
    g = g / np.linalg.norm(g)
    print(f"  Gender direction computed from {len(diffs)} pairs. "
          f"PC1 explains {pca.explained_variance_ratio_[0]:.1%} of variance.")
    return g


def cosine(a, b):
    a, b = np.array(a, dtype=float), np.array(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 1e-8 else 0.0

## 5. Identify Changed Words Between Sentence Pairs

For each CrowS-Pairs example, we find the words that differ between `sent_more` and `sent_less`. These are the key tokens that carry the stereotypic vs anti-stereotypic content.

In [ ]:
import re

def tokenize_words(sentence):
    """Simple word tokenizer — lowercased, punctuation stripped."""
    return re.findall(r"\b[a-zA-Z]+\b", sentence.lower())


def get_changed_words(sent_more, sent_less):
    """
    Returns (words_only_in_more, words_only_in_less).
    Uses set difference on words — captures words unique to each sentence.
    """
    words_more = set(tokenize_words(sent_more))
    words_less = set(tokenize_words(sent_less))

    # Stop words to exclude (articles, prepositions, etc.)
    STOP = {"the", "a", "an", "is", "was", "are", "were", "be", "been", 
            "in", "on", "at", "to", "of", "and", "or", "but", "it",
            "he", "she", "they", "his", "her", "their", "i", "we"}

    only_more = (words_more - words_less) - STOP
    only_less = (words_less - words_more) - STOP

    return list(only_more), list(only_less)


# Preview changed words on a few examples
print("Sample changed words:")
for i in range(3):
    ex = crows_gender[i]
    more_w, less_w = get_changed_words(ex["sent_more"], ex["sent_less"])
    print(f"  Example {i+1}:")
    print(f"    sent_more: {ex['sent_more'][:80]}")
    print(f"    sent_less: {ex['sent_less'][:80]}")
    print(f"    Words unique to sent_more: {more_w}")
    print(f"    Words unique to sent_less: {less_w}")
    print()

## 6. Geometric SPS — Bolukbasi on CrowS-Pairs

For each sentence pair:
1. Find the changed words (words unique to `sent_more` vs `sent_less`)
2. Get contextual representations of those words
3. Compute their projection onto the gender direction **g**
4. If the `sent_more` words project **more** onto g → model's geometry favours the stereotyped direction

$$\text{Geometric SPS} = \frac{1}{|P|} \sum_{(s_+, s_-) \in P} \mathbb{1}\left[|\cos(\bar{v}_{s_+},\ g)| > |\cos(\bar{v}_{s_-},\ g)|\right] \times 100$$

Where $\bar{v}_s$ is the mean contextual representation of the changed words in sentence $s$.

In [ ]:
def compute_geometric_sps(model, tokenizer, g, dataset, desc=""):
    """
    Geometric SPS: % of pairs where changed words in sent_more project
    more strongly onto the gender direction than changed words in sent_less.
    """
    geo_prefer_stereo = 0
    valid_pairs       = 0
    records           = []

    for ex in tqdm(dataset, desc=f"Geometric SPS {desc}"):
        more_words, less_words = get_changed_words(ex["sent_more"], ex["sent_less"])

        # Need at least one changed word on each side
        if not more_words or not less_words:
            continue

        # Average contextual rep of changed words on each side
        more_vecs = [get_contextual_rep(model, tokenizer, w) for w in more_words[:3]]
        less_vecs = [get_contextual_rep(model, tokenizer, w) for w in less_words[:3]]

        v_more = np.mean(more_vecs, axis=0)
        v_less = np.mean(less_vecs, axis=0)

        proj_more = abs(cosine(v_more, g))
        proj_less = abs(cosine(v_less, g))

        geo_prefers = proj_more > proj_less
        geo_prefer_stereo += int(geo_prefers)
        valid_pairs += 1

        records.append({
            "more_words":   more_words,
            "less_words":   less_words,
            "proj_more":    round(proj_more, 4),
            "proj_less":    round(proj_less, 4),
            "geo_prefers":  int(geo_prefers),
        })

    geo_sps = 100 * geo_prefer_stereo / valid_pairs if valid_pairs > 0 else 0
    print(f"Geometric SPS {desc}: {geo_sps:.2f}%  ({geo_prefer_stereo}/{valid_pairs} valid pairs)")
    print(f"  Ideal = 50.00%  |  Bias above baseline = {geo_sps - 50:.2f}pp")
    return geo_sps, records

## 7. Load Models & Run All Metrics

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token

crows_results = {}
CONDITIONS = ["Baseline", "Post-LoRA", "Post-QLoRA"]

In [ ]:
# --- CONDITION 1: Baseline ---
print("\n" + "="*55)
print("CONDITION: Baseline")
print("="*55)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
base_model.eval()

sps_score, sps_records = compute_sps(base_model, tokenizer, crows_gender, desc="(Baseline)")

print("\nComputing gender direction for Baseline...")
g_base = compute_gender_direction(base_model, tokenizer, GENDER_PAIRS)

geo_sps, geo_records = compute_geometric_sps(base_model, tokenizer, g_base, crows_gender, desc="(Baseline)")

crows_results["Baseline"] = {"sps": sps_score, "geo_sps": geo_sps,
                              "sps_records": sps_records, "geo_records": geo_records}
del base_model
torch.cuda.empty_cache()

In [ ]:
# --- CONDITION 2: Post-LoRA ---
print("\n" + "="*55)
print("CONDITION: Post-LoRA")
print("="*55)
base_model  = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
lora_model  = PeftModel.from_pretrained(base_model, LORA_PATH)
merged_lora = lora_model.merge_and_unload()
merged_lora.eval()

sps_score, sps_records = compute_sps(merged_lora, tokenizer, crows_gender, desc="(Post-LoRA)")

print("\nComputing gender direction for Post-LoRA...")
g_lora = compute_gender_direction(merged_lora, tokenizer, GENDER_PAIRS)

geo_sps, geo_records = compute_geometric_sps(merged_lora, tokenizer, g_lora, crows_gender, desc="(Post-LoRA)")

crows_results["Post-LoRA"] = {"sps": sps_score, "geo_sps": geo_sps,
                               "sps_records": sps_records, "geo_records": geo_records}
del merged_lora
torch.cuda.empty_cache()

In [ ]:
# --- CONDITION 3: Post-QLoRA ---
print("\n" + "="*55)
print("CONDITION: Post-QLoRA")
print("="*55)
bnb_config  = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
quant_base  = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
qlora_model = PeftModel.from_pretrained(quant_base, QLORA_PATH)
qlora_model.eval()

sps_score, sps_records = compute_sps(qlora_model, tokenizer, crows_gender, desc="(Post-QLoRA)")

print("\nComputing gender direction for Post-QLoRA...")
g_qlora = compute_gender_direction(qlora_model, tokenizer, GENDER_PAIRS)

geo_sps, geo_records = compute_geometric_sps(qlora_model, tokenizer, g_qlora, crows_gender, desc="(Post-QLoRA)")

crows_results["Post-QLoRA"] = {"sps": sps_score, "geo_sps": geo_sps,
                                "sps_records": sps_records, "geo_records": geo_records}
del qlora_model
torch.cuda.empty_cache()

## 8. Results: SPS + Geometric SPS

In [ ]:
# Summary table
summary = {
    cond: {
        "SPS (%)": round(crows_results[cond]["sps"], 2),
        "SPS bias above 50%": round(crows_results[cond]["sps"] - 50, 2),
        "Geometric SPS (%)": round(crows_results[cond]["geo_sps"], 2),
        "Geo SPS bias above 50%": round(crows_results[cond]["geo_sps"] - 50, 2),
    }
    for cond in CONDITIONS
}

df = pd.DataFrame(summary).T
print("\n" + "="*65)
print("CROWS-PAIRS RESULTS")
print("(SPS ideal = 50% — higher = more biased)")
print("="*65)
print(df.to_string())

df.to_csv("../results/crows_pairs_results.csv")

# Save full results
save_data = {
    cond: {
        "sps":     crows_results[cond]["sps"],
        "geo_sps": crows_results[cond]["geo_sps"],
    }
    for cond in CONDITIONS
}
with open("../results/crows_pairs_results.json", "w") as f:
    json.dump(save_data, f, indent=2)
print("\nSaved to ../results/crows_pairs_results.json")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ["#4C72B0", "#DD8452", "#55A868"]

sps_vals     = [crows_results[c]["sps"]     for c in CONDITIONS]
geo_sps_vals = [crows_results[c]["geo_sps"] for c in CONDITIONS]

# Panel 1: Behavioural SPS
ax = axes[0]
bars = ax.bar(CONDITIONS, sps_vals, color=colors, width=0.45)
ax.axhline(50, color="red", linestyle="--", linewidth=1.2, label="Ideal (50%)")
ax.set_ylim(0, 100)
ax.set_ylabel("SPS (%)")
ax.set_title("Stereotype Preference Score (SPS)\nBehavioural — Log-Likelihood Scoring", fontsize=11)
ax.legend(fontsize=9)
for bar, v in zip(bars, sps_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.8, f"{v:.1f}%", ha="center", fontsize=11)

# Panel 2: Geometric SPS
ax = axes[1]
bars = ax.bar(CONDITIONS, geo_sps_vals, color=colors, width=0.45)
ax.axhline(50, color="red", linestyle="--", linewidth=1.2, label="Ideal (50%)")
ax.set_ylim(0, 100)
ax.set_ylabel("Geometric SPS (%)")
ax.set_title("Geometric SPS (Bolukbasi on CrowS-Pairs)\n% of pairs where stereo words project more onto g", fontsize=11)
ax.legend(fontsize=9)
for bar, v in zip(bars, geo_sps_vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.8, f"{v:.1f}%", ha="center", fontsize=11)

plt.suptitle("CrowS-Pairs Gender Bias — Baseline vs Post-LoRA vs Post-QLoRA", fontsize=12)
plt.tight_layout()
plt.savefig("../results/crows_pairs_sps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to ../results/crows_pairs_sps.png")

## 9. Agreement Between SPS and Geometric SPS

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

x = np.arange(len(CONDITIONS))
width = 0.35

b1 = ax.bar(x - width/2, sps_vals,     width, label="Behavioural SPS",  color="#4C72B0")
b2 = ax.bar(x + width/2, geo_sps_vals, width, label="Geometric SPS",    color="#DD8452")
ax.axhline(50, color="red", linestyle="--", linewidth=1, label="Ideal (50%)")

ax.set_xticks(x)
ax.set_xticklabels(CONDITIONS)
ax.set_ylim(0, 100)
ax.set_ylabel("Score (%)")
ax.set_title("Behavioural SPS vs Geometric SPS\n(Do surface scores align with embedding geometry?)")
ax.legend()

plt.tight_layout()
plt.savefig("../results/sps_vs_geometric_sps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to ../results/sps_vs_geometric_sps.png")

## 10. Write-Up

### Dataset
CrowS-Pairs (Nangia et al., 2020) contains 1,508 minimal pairs of sentences — one more stereotyping (`sent_more`) and one less stereotyping (`sent_less`) — across 9 bias categories. We focus on the **gender** subset.

### Task 2 — SPS (Stereotype Preference Score)

**Definition:** % of sentence pairs where the model assigns higher per-token log-likelihood to the more-stereotyping sentence. Ideal unbiased model = 50%.

> CrowS-Pairs was designed for masked LMs using pseudo-log-likelihood (MLM scoring). For OPT-1.3B (a causal decoder LM), we use standard causal log-likelihood normalised by token count — a well-established adaptation.

| Condition | SPS | Bias above 50% |
|---|---|---|
| Baseline | *fill in* | *fill in* |
| Post-LoRA | *fill in* | *fill in* |
| Post-QLoRA | *fill in* | *fill in* |

### Task 3 — Geometric SPS (Bolukbasi on CrowS-Pairs)

**Definition:** For each sentence pair, we identify the words that differ between `sent_more` and `sent_less`. We extract their contextual representations (final hidden layer) and project them onto the gender direction **g** (computed via PCA of 15 male–female difference vectors). Geometric SPS = % of pairs where the changed words in `sent_more` project more strongly onto g.

This bridges Bolukbasi's geometric framework with CrowS-Pairs' sentence-level design — asking whether the *embedding geometry* of the stereotypic words reflects the behavioural preference captured by SPS.

### Observations
*Fill in after running.*

### Key Comparison: Behavioural SPS vs Geometric SPS

If both metrics move in the same direction across fine-tuning conditions, it suggests that surface-level bias (what the model *outputs*) is anchored to geometric bias (how the model *represents* gender internally). If they diverge, it suggests a dissociation — the model's output behaviour may have changed while its internal geometry remains biased, or vice versa.